In [ ]:
import pandas as pd
import numpy as np


def find_closest_match(csv_file, input_labels):
    """
    Finds the closest match in a database based on the specified labels using mean absolute error (MAE).

    Args:
        csv_file (str): Path to the CSV file containing the database.
        input_labels (dict): Dictionary containing the 5 labels: color, length, height, depth, and price.
                             Example: {'color': 'White', 'length': 50, 'height': 30, 'depth': 40, 'price': 300}

    Returns:
        pd.Series: Closest matching row from the database.
    """
    # Load the CSV file into a DataFrame
    df = pd.read_csv(csv_file)

    # Ensure the input labels have all required keys
    required_keys = {'color', 'length', 'height', 'depth', 'price'}
    if not required_keys.issubset(input_labels.keys()):
        raise ValueError(f"Input labels must contain the keys: {required_keys}")

    # Filter the DataFrame to include only rows with the same color
    color_filtered_df = df[df['Color'].str.lower() == input_labels['color'].lower()]

    # If no rows match the color, return None
    if color_filtered_df.empty:
        return None

    # Compute MAE for numerical columns (length, height, depth, price)
    numerical_columns = ['Length (m)', 'Height (cm)', 'Depth (m)', 'Price (USD)']
    mae_values = color_filtered_df[numerical_columns].apply(
        lambda row: np.mean([
            abs(row['Length (m)'] - input_labels['length']),
            abs(row['Height (cm)'] - input_labels['height']),
            abs(row['Depth (m)'] - input_labels['depth']),
            abs(row['Price (USD)'] - input_labels['price'])
        ]), axis=1
    )

    # Find the row with the smallest MAE
    best_match_index = mae_values.idxmin()
    best_match = color_filtered_df.loc[best_match_index]

    return best_match


In [ ]:
# Example CSV file path
csv_file_path = 'dining_and_desk_data.csv'

# Example input labels
input_labels = {
    'color': 'White',
    'length': 59.06,  # Example length in inches
    'height': 29.53,  # Example height in inches
    'depth': 35.43,   # Example depth in inches
    'price': 300      # Example price in USD
}

# Find the closest match
closest_match = find_closest_match(csv_file_path, input_labels)

# Print the result
if closest_match is not None:
    print("Closest match found:")
    print(closest_match)
else:
    print("No match found for the given color.")


**New**

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
def convert_sizes_to_inch(json_input):
    # Conversion factors to inches
    conversion_factors = {
        "ft": 12,       # 1 foot = 12 inches
        "cm": 0.393701, # 1 cm = 0.393701 inches
        "m": 39.3701,   # 1 meter = 39.3701 inches
        "inch": 1       # 1 inch = 1 inch
    }

    def convert_size(size_value):
        for unit, factor in conversion_factors.items():
            if size_value.endswith(unit):
                # Extract the numeric value and convert it to inches
                numeric_value = float(size_value.replace(unit, "").strip())
                return round(numeric_value * factor, 2)  # Round to 2 decimal places
        return float(size_value)  # Return None if no valid un   it is found

    data = json_input

    # Iterate through keys and convert size-related values to inches
    for key in ["length", "width", "height"]:
        if key in data and isinstance(data[key], str):  # Check if the key exists and is a string
            converted_value = convert_size(data[key])
            if converted_value is not None:
                data[key] = converted_value

    # Return the modified JSON as a string
    return data

In [5]:
# Load a pre-trained SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def color_embedding_function(color_name):
  return model.encode(color_name)

def find_top_matches(csv_file, inputs, color_embedding_function, n=1, weights=None):
    """
    Finds the top N closest matches in a database based on the specified labels using a weighted scoring system.

    Args:
        csv_file (str): Path to the CSV file containing the database.
        inputs (dict): Dictionary containing labels (e.g., color, length, height, depth, and price).
        color_embedding_function (callable): Function to convert color labels into embeddings.
        n (int): Number of top matches to return.
        weights (dict): Weights for each component in the matching process (default: equal weights).
                        Example: {'color': 0.4, 'dimensions': 0.4, 'price': 0.2}

    Returns:
        pd.DataFrame: DataFrame containing the top N matches with their table_id and scores.
    """
    # Load the CSV file into a DataFrame
    df = pd.read_csv(csv_file)

    # Define default weights if none are provided
    if weights is None:
        weights = {'color': 0.4, 'dimensions': 0.4, 'price': 0.2}

    # Initialize scores
    df['color_score'] = 0
    df['dimension_score'] = 0
    df['price_score'] = 0
    df['final_score'] = 0

    # Color Matching
    if 'color' in inputs:
        input_color_embedding = color_embedding_function(inputs['color'].lower())
        df['color_score'] = df['color'].apply(
            lambda db_color: cosine_similarity(
                [input_color_embedding],
                [color_embedding_function(db_color)]
            )[0][0]
        )

    # Dimension Matching (handles missing dimensions)
    dimension_keys = ['length', 'height', 'width']
    input_dimensions = {key: inputs[key] for key in dimension_keys if key in inputs}

    if input_dimensions:
        for key in input_dimensions:
            df[f'{key}_diff'] = df[key].apply(lambda x: abs(x - input_dimensions[key]))
        max_diffs = df[[f'{key}_diff' for key in input_dimensions]].max(axis=0)
        df['dimension_score'] = 1 - df[[f'{key}_diff' for key in input_dimensions]].sum(axis=1) / max_diffs.sum()

    # Price Matching
    if 'price' in inputs:
        max_price_diff = df['price'].max() - df['price'].min()
        df['price_score'] = df['price'].apply(
            lambda x: 1 - abs(x - inputs['price']) / max_price_diff
        )

    # Calculate Weighted Final Score
    df['final_score'] = (
        weights['color'] * df['color_score'] +
        weights['dimensions'] * df['dimension_score'] +
        weights['price'] * df['price_score']
    )

    # Sort by final score in descending order and get the top N matches
    top_matches = df.nlargest(n, 'final_score')[['img_dir', 'price', 'link', 'final_score']]

    return top_matches if not top_matches.empty else None

In [7]:
# Sample input labels
input_labels = {
    'color': 'black',
    'length': "5 ft",
    'height': "3 ft" ,
    'price': 500
}

# Weights (optional)
weights = {'color': 0.5, 'dimensions': 0.3, 'price': 0.2}
data_path = "/content/gdrive/MyDrive/ECE1786_project/product_database.csv"

In [8]:
# Call the function
inputs = convert_sizes_to_inch(input_labels)
print(inputs)
best_match = find_top_matches(data_path, inputs, color_embedding_function, n=5, weights=weights)

print("Best Match:\n", best_match)

{'color': 'black', 'length': 60.0, 'height': 36.0, 'price': 500}
Best Match:
      img_dir   price                                               link  \
139      140  291.65  https://www.temu.com/ca/-63-large-dining-table...   
73        74  570.98  https://www.amazon.ca/HSH-Farmhouse-Computer-E...   
149      150  181.99  https://www.temu.com/ca/outdoor-metal-bar--pat...   
100      101  269.99  https://www.amazon.ca/COSTWAY-Rectangular-Farm...   
112      113  269.99  https://www.amazon.ca/COSTWAY-Rectangular-Farm...   

     final_score  
139     0.972444  
73      0.972438  
149     0.968106  
100     0.968099  
112     0.968099  


In [ ]:
input_labels['color'].lower()

'light blue'

In [ ]:
json_input = {"color": "Dark", "length": "4 ft", "width": "120 cm", "height": "3 m", "price": 50}
convert_sizes_to_inch(json_input)

'{"color": "Dark", "length": 48.0, "width": 47.24, "height": 118.11, "price": 50}'